# EEG Pipeline Development Notebook

Development notebook for testing the unified preprocessing pipeline with TUH and HBN datasets.

In [ ]:
import mne
import os
from pathlib import Path
import numpy as np
import h5py

from speed.pipeline import BasePipeline, PretrainPipeline
from speed.utils import make_tuh_montage
from speed.methods import PreprocessMethods

## Preprocessing pipeline

In [ ]:
# raw_path  = "/data/agjma/HBN_EEG_Raw/ds005505/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-RestingState_eeg.set" 
# raw_path = "/dtu-compute/EEG_at_scale/tuh_eeg/data/tuh_eeg/v2.0.0/edf/066/aaaaajwo/s002_2010_06_03/01_tcp_ar/aaaaajwo_s002_t000.edf"
raw_path = "/dtu-compute/EEG_at_scale/tuh_eeg/data/tuh_eeg/v2.0.0/edf/066/aaaaajwo/s002_2010_06_03/01_tcp_ar/aaaaajwo_s002_t001.edf"
raw = mne.io.read_raw_edf(raw_path, preload=True)

In [ ]:
# raw = mne.io.read_raw_eeglab(raw_path, preload=True)
# print(raw.ch_names)
hbn_channels = ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'E25', 'E26', 'E27', 'E28', 'E29', 'E30', 'E31', 'E32', 'E33', 'E34', 'E35', 'E36', 'E37', 'E38', 'E39', 'E40', 'E41', 'E42', 'E43', 'E44', 'E45', 'E46', 'E47', 'E48', 'E49', 'E50', 'E51', 'E52', 'E53', 'E54', 'E55', 'E56', 'E57', 'E58', 'E59', 'E60', 'E61', 'E62', 'E63', 'E64', 'E65', 'E66', 'E67', 'E68', 'E69', 'E70', 'E71', 'E72', 'E73', 'E74', 'E75', 'E76', 'E77', 'E78', 'E79', 'E80', 'E81', 'E82', 'E83', 'E84', 'E85', 'E86', 'E87', 'E88', 'E89', 'E90', 'E91', 'E92', 'E93', 'E94', 'E95', 'E96', 'E97', 'E98', 'E99', 'E100', 'E101', 'E102', 'E103', 'E104', 'E105', 'E106', 'E107', 'E108', 'E109', 'E110', 'E111', 'E112', 'E113', 'E114', 'E115', 'E116', 'E117', 'E118', 'E119', 'E120', 'E121', 'E122', 'E123', 'E124', 'E125', 'E126', 'E127', 'E128', 'Cz']
tuh_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']

In [ ]:
# TUH pipeline (standard preprocessing)
pipeline = PretrainPipeline(
    do_ica=False, 
    line_freqs=[60], 
    channels_rename=None, 
    montage="tuh",  # "tuh", standard MNE name, or path to .fif
    channels=tuh_channels, 
    window_length=30, 
    shift_seconds=None,  # None = no overlap
    hp_freq=0.5,
    lp_freq=50,
    sfreq=100,
    return_quality_metrics=False,
    drop_bad_quality=True,
    standardize_channel_names=True,  # TUH-style channel name standardization
)

In [ ]:
raws, times, indices = pipeline([raw_path], )

In [ ]:
# TUH pipeline with target montage interpolation
# Use target_montage to interpolate all data to a specific electrode layout
pipeline_interp = PretrainPipeline(
    do_ica=False, 
    line_freqs=[60], 
    channels_rename=None, 
    montage="tuh",
    channels=tuh_channels, 
    window_length=30, 
    shift_seconds=None,
    hp_freq=0.5,
    lp_freq=50,
    sfreq=100,
    return_quality_metrics=False,
    drop_bad_quality=True,
    standardize_channel_names=True,
    target_montage="montage-hbn19-dig.fif",  # path to target montage .fif
)

raws, times, indices = pipeline_interp([raw_path])

## Check manually if file looks okay: 


In [ ]:
data1_path = "/scratch/linsk/tuh_test_preprocessed/data_1.hdf5"
with h5py.File(data1_path, 'r') as f:
    # print(dict(f.attrs).keys())
    data = f['data'][:]
    print(data.shape) # (windows, 105, 3000)
    print(f.attrs["files"])

In [ ]:
import mne

window_idx = 0
eeg_data = data[window_idx] 

sfreq = 100 
ch_names = [f"Ch{i}" for i in range(eeg_data.shape[0])]
ch_types = ["eeg"] * eeg_data.shape[0]

info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
raw = mne.io.RawArray(eeg_data, info)

raw.plot(scalings='auto', title=f"Window {window_idx}", show=True)
;


## Quality metrics inspection
Use the cells below to inspect the quality metrics log when troubleshooting why windows were or were not dropped.

In [ ]:
import pandas as pd 
log_file = "/scratch/linsk/tuh_test_preprocessed/logs/quality_metrics.csv"
df = pd.read_csv(log_file, sep=",", header=0)

In [ ]:
# get above file name: 
df['filename'][0]

In [ ]:
df

### Quality Metrics (configurable via `*_limit` params)

| Metric | Default Limit | Description |
|--------|---------------|-------------|
| **oha** | < 0.8 | Overall High Amplitude - frac samples exceeding `oha_threshold` |
| **thv** | < 0.5 | Temporal High Variance - frac time points with high std |
| **chv** | < 0.5 | Channel High Variance - frac channels with high std |
| **bcr** | < 0.8 | Bad Channel Ratio - frac of bad/discrete channels |

## Align montages: 


In [ ]:
from mne.channels import make_dig_montage
# make_tuh_montage, PreprocessMethods already imported at top

In [ ]:
raw_path_hbn  = "/scratch/linsk/hbn_test/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-RestingState_eeg.set"
raw_path_tuh = "/scratch/linsk/tuh_test/aaaaabay/s001_2003_05_20/02_tcp_le/aaaaabay_s001_t000.edf"

raw_hbn = mne.io.read_raw_eeglab(raw_path_hbn, preload=True)
raw_tuh = mne.io.read_raw_edf(raw_path_tuh, preload=True)

In [ ]:
def get_channel_positions(raw):
    """Return a dict {ch_name: xyz} for channels that have locations."""
    positions = {}
    for ch in raw.info['chs']:
        ch_name = ch['ch_name']
        loc = ch['loc'][:3]  # first 3 elements are x, y, z
        if np.any(loc):  # skip if all zeros
            positions[ch_name] = loc
    return positions

In [ ]:
positions_hbn = {ch: pos for ch, pos in get_channel_positions(raw_hbn).items()}
montage_hbn = make_dig_montage(ch_pos=positions_hbn, coord_frame='head')
montage_gsn = mne.channels.make_standard_montage('GSN-HydroCel-128')
montage_10_20 = mne.channels.make_standard_montage('standard_1020')

montage_tuh = make_tuh_montage()
PreprocessMethods.to_standard_names(raw_tuh)
PreprocessMethods.set_montage(raw_tuh, montage_tuh);

### Get HBN montage:

In [ ]:
tuh_to_hbn_dict = {'Fp1': 'E25',
 'Fp2': 'E8',
 'F3': 'E24',
 'F4': 'E124',
 'C3': 'E36',
 'C4': 'E104',
 'P3': 'E60',
 'P4': 'E85',
 'O1': 'E70',
 'O2': 'E83',
 'F7': 'E33',
 'F8': 'E122',
 'T7': 'E45',
 'T8': 'E108',
 'T5': 'E58',
 'T6': 'E96',
 'Fz': 'E5',
 'Cz': 'Cz',
 'Pz': 'E62'}


In [ ]:
# hbn_montage script: 
raw_path_hbn  = "/scratch/linsk/hbn_test/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-RestingState_eeg.set"
raw_hbn = mne.io.read_raw_eeglab(raw_path_hbn, preload=True)
positions_hbn = {ch['ch_name']: ch['loc'][:3] for ch in raw_hbn.info['chs'] if np.any(ch['loc'][:3])}

positions_hbn_filtered = {ch: loc for ch, loc in positions_hbn.items() if ch in tuh_to_hbn_dict.values()}

montage_hbn = make_dig_montage(ch_pos=positions_hbn, coord_frame='head')
montage_hbn.save('montage-hbn128-dig.fif', overwrite= True)

montage_hbn2 = mne.channels.read_dig_fif('montage-hbn-dig.fif')
# raw_hbn.set_montage(montage_hbn2)

# raw_path_tuh = "/scratch/linsk/tuh_test/aaaaabay/s001_2003_05_20/02_tcp_le/aaaaabay_s001_t000.edf"
# raw_tuh = mne.io.read_raw_edf(raw_path_tuh, preload=True)
# montage_tuh = make_tuh_montage()
# PreprocessMethods.to_standard_names(raw_tuh)
# PreprocessMethods.set_montage(raw_tuh, montage_tuh)


# raw = raw_tuh.interpolate_to(sensors=montage_hbn2, method='spline')

In [ ]:
montage_hbn2

### Montage plots

In [ ]:
# combined_montage.plot();

    # fig = combined_montage.plot(kind='3d', show_names=True)
    # fig.suptitle("Combined Montage (20 + 20 channels)", fontsize=14)

ten20 = mne.channels.make_standard_montage('standard_1020')
ten20.plot();

gsn = mne.channels.make_standard_montage('GSN-HydroCel-128')

In [ ]:
import matplotlib.pyplot as plt

def plot_two_montages_2d(montage1, montage2, color1='blue', color2='red'):
    pos1 = montage1.get_positions()['ch_pos']
    pos2 = montage2.get_positions()['ch_pos']

    fig, ax = plt.subplots(figsize=(6, 6))

    # Montage 
    # Montage 1
    for ch, pos in pos1.items():
        ax.scatter(pos[0], pos[1], color=color1, label='Montage 1' if ch == list(pos1.keys())[0] else "", alpha=0.7)
        ax.text(pos[0], pos[1], ch, fontsize=8, color=color1, ha='center')

    # Montage 2
    for ch, pos in pos2.items():
        ax.scatter(pos[0], pos[1], color=color2, label='Montage 2' if ch == list(pos2.keys())[0] else "", alpha=0.7)
        ax.text(pos[0], pos[1], ch, fontsize=8, color=color2, ha='center')

    ax.legend()
    ax.set_title("Overlay of Two Montages (2D projection)")
    ax.set_xlabel("X position")
    ax.set_ylabel("Y position")
    ax.axis('equal')
    plt.show()



In [ ]:
montage1 = mne.channels.make_standard_montage("biosemi128")
montage2 = mne.channels.make_standard_montage("standard_1020")

plot_two_montages_2d(montage1, montage2, color1='blue', color2='red')


In [ ]:
montage1 = mne.channels.make_standard_montage("GSN-HydroCel-128")
montage2 = mne.channels.make_standard_montage("standard_1020")

plot_two_montages_2d(montage1, montage2, color1='blue', color2='red')

In [ ]:
plot_two_montages_2d(montage_tuh, montage_hbn, color1='blue', color2='red')

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

def plot_two_montages_2d_unit(montage1, montage2, color1='blue', color2='red'):
    # get positions
    pos1 = np.array(list(montage1.get_positions()['ch_pos'].values()))
    pos2 = np.array(list(montage2.get_positions()['ch_pos'].values()))
    
    # normalize to unit vectors (remove scale)
    pos1_uv = pos1 / np.linalg.norm(pos1, axis=1)[:, None]
    pos2_uv = pos2 / np.linalg.norm(pos2, axis=1)[:, None]
    
    # project to 2D (simple orthographic)
    def proj(xyz):
        return xyz[:, 0], xyz[:, 1]
    
    x1, y1 = proj(pos1_uv)
    x2, y2 = proj(pos2_uv)
    
    plt.figure(figsize=(6,6))
    plt.scatter(x1, y1, c=color1, label='GSN-HydroCel-128')
    plt.scatter(x2, y2, c=color2, label='10-20')
    plt.axis('equal')
    plt.legend()
    plt.show()

# Example usage
montage1 = mne.channels.make_standard_montage("GSN-HydroCel-128")
montage2 = mne.channels.make_standard_montage("standard_1020")
plot_two_montages_2d_unit(montage1, montage2)


### mapping: rotation + translation from hbn to tuh 

*WIP: Montage alignment experiments*

In [ ]:
from mne.coreg import fit_matched_points
from mne.transforms import apply_trans
from mne.transforms import Transform

from mne.transforms import read_trans
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import cKDTree


In [ ]:
fids_tuh = raw_tuh.info['dig'][:3]
fids_hbn = raw_hbn.info['dig'][:3]

fids_hbn_arr = np.array([f['r'] for f in fids_hbn], dtype=float)
fids_tuh_arr = np.array([f['r'] for f in fids_tuh], dtype=float)

# add cz to both:
fids_tuh_arr = np.vstack([fids_tuh_arr, get_channel_positions(raw_tuh)['Cz']])
fids_hbn_arr = np.vstack([fids_hbn_arr, get_channel_positions(raw_hbn)['Cz']])

In [ ]:
trans = fit_matched_points(fids_hbn_arr, fids_tuh_arr, 'similarity')  # map montage2 → montage1

In [ ]:
# load raw_tuh, then:
# montage_tuh = make_tuh_montage()
# PreprocessMethods.to_standard_names(raw_tuh)
# PreprocessMethods.set_montage(raw_tuh, montage_tuh);

tuh_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']
positions_tuh = get_channel_positions(raw_tuh) # 24?
positions_tuh = {k:v for k, v in positions_tuh.items() if k in tuh_channels} # should only be 19 now 
# all_positions_tuh = montage_tuh.get_positions()['ch_pos'] # full 345?
montage_tuh_subset = make_dig_montage(ch_pos=positions_tuh, coord_frame='head')

#### below code to create .fif file that can be loaded into mne coreg gui for manual alignment

In [ ]:
# for d, name in zip(montage_tuh_subset.dig, tuh_channels):
#     d['ch_name'] = name

# montage_tuh_subset._coord_frame = 'head'
# montage_tuh_subset.save('montage-tuh-dig.fif', overwrite = True)
# # [d.get('ch_name') for d in montage_tuh_subset.dig] # check if names exist

In [ ]:
# # Create dummy Raw with only channel names
# ch_names = [d['ch_name'] for d in montage_tuh_subset.dig]
# info = mne.create_info(ch_names=ch_names, sfreq=1000., ch_types='eeg')

# # Set montage
# raw_dummy = mne.io.RawArray(np.zeros((len(ch_names), 1)), info)
# raw_dummy.set_montage(montage_tuh_subset)

# # Save
# raw_dummy.save('tuh_raw_dummy.fif')

#### plot if above alignments worked

In [ ]:
trans = read_trans('tuh_to_hbn2-trans.fif')

In [ ]:
# Extract electrode positions
montage_pos = np.array([d['r'] for d in montage_tuh_subset.dig])

# Apply rigid head->head transform manually
aligned_pos = apply_trans(trans, montage_pos)

# Update the montage
for d, r_new in zip(montage_tuh_subset.dig, aligned_pos):
    d['r'] = r_new


In [ ]:
# Example: TUH (aligned) and HBN template
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# TUH montage (after alignment)
tuh_pos = np.array([d['r'] for d in montage_tuh_subset.dig])
ax.scatter(tuh_pos[:,0], tuh_pos[:,1], tuh_pos[:,2], c='r', label='TUH')

# HBN template
hbn_pos = np.array([d['r'] for d in montage_hbn.dig])
ax.scatter(hbn_pos[:,0], hbn_pos[:,1], hbn_pos[:,2], c='b', label='HBN')

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.legend()
plt.show()


In [ ]:
R = trans[:3, :3]
scale = np.cbrt(np.linalg.det(R))  # approximate uniform scale
print("Scale factor applied:", scale)



Duplicate scale-factor calculation removed (see previous cell).

### get closest electrodes after alignment

In [ ]:
# TUH electrode positions after rigid alignment
tuh_pos = np.array([d['r'] for d in montage_tuh_subset.dig])


In [ ]:
tuh_pos

In [ ]:
# HBN electrode positions (template)
hbn_pos = np.array([d['r'] for d in montage_hbn.dig])
hbn_ch_names = list(montage_hbn.get_positions()['ch_pos'].keys())

In [ ]:
# Build KD-tree on HBN template
tree = cKDTree(hbn_pos)

# Find closest HBN electrode for each TUH electrode
dist, idx = tree.query(tuh_pos)
closest_hbn_chs = [hbn_ch_names[i] for i in idx] # list
tuh_to_hbn = {tuh: hbn_ch_names[i] for tuh, i in zip(positions_tuh.keys(), idx)} # dict


In [ ]:
tuh_to_hbn

### Plot to verify (2D)
Scatter plot comparing aligned TUH and HBN electrode layouts.

In [ ]:
import matplotlib.pyplot as plt

# XY positions
tuh_xy = tuh_pos[:, :2]
hbn_xy = hbn_pos[:, :2]

plt.figure(figsize=(8,8))
plt.scatter(hbn_xy[:,0], hbn_xy[:,1], c='blue', label='HBN')
plt.scatter(tuh_xy[:,0], tuh_xy[:,1], c='red', label='TUH')

# Add TUH labels
for i, name in enumerate(positions_tuh.keys()):
    plt.text(tuh_xy[i,0]+0.002, tuh_xy[i,1]+0.002, name, color='red', fontsize=8)

# Add HBN labels
for i, name in enumerate(hbn_ch_names):
    plt.text(hbn_xy[i,0]+0.002, hbn_xy[i,1]+0.002, name, color='blue', fontsize=6)

plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.title('TUH vs HBN electrodes (top-down)')
plt.legend()
plt.axis('equal')
plt.show()
